# Pamoka 10 - Dirbtinio intelekto agentai gamyboje

Šioje pamokoje sužinosite apie **gamybinius šablonus** dirbtinio intelekto agentams, naudojant Microsoft Agent Framework (Python). Aptarsime:

- **Observabilumas** — pridėti laiko matavimą ir žurnalavimą agentų sąveikoms
- **Įvertinimas** — naudojant įvertinimo agentą atsakymų kokybei įvertinti
- **Sąnaudų valdymas** — strategijos tokenų optimizavimui ir modelio pasirinkimui

Scenarijus — **kelionių agentas**, padedantis vartotojams planuoti keliones, su papildoma stebėsena ir vertinimu.


## Nustatymai


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Gamybiniai aspektai

Perkėlimas DI agentų iš prototipų į gamybą reikalauja atidaus dėmesio trims kertinėms sritims:

1. **Observabilumas** — Turite matomumą, ką agentas daro, kiek laiko tai užtrunka ir kokius įrankius jis kviečia. Be trasavimo ir registravimo, gamybinių problemų derinimas yra beveik neįmanomas.

2. **Vertinimas** — Automatiniai kokybės patikrinimai užtikrina, kad agento atsakymai laikui bėgant išliktų tikslūs, pilni ir naudingi. Vertinimo agentas gali įvertinti atsakymus pagal apibrėžtus kriterijus.

3. **Sąnaudų valdymas** — Žetonų naudojimas tiesiogiai įtakoja išlaidas. Strategijos, tokios kaip užklausų optimizavimas, modelio pasirinkimas ir kešavimas, padeda kontroliuoti išlaidas neaukojant kokybės.


## Stebimo agento kūrimas

Apibrėžiame kelionės įrankius ir apgaubiam agento kvietimą laiko matavimo funkcija, kad galėtume stebėti delsą. Gamybinėje aplinkoje integruotumėte OpenTelemetry arba panašų sekimo backend'ą.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Vertinimo modeliai

Įprasta gamybinė schema yra naudoti antrą agentą kaip **vertintoją**. Vertintojas įvertina pagrindinio agento atsakymą pagal iš anksto nustatytus kriterijus, tokius kaip išsamumas, tikslumas ir naudingumas.

Tai leidžia:
- Automatizuoti kokybės patikrinimai prieš atsakymų pateikimą vartotojams
- Regresijų aptikimas, kai keičiasi užklausos arba modeliai
- Nuolatinis agento veiklos stebėjimas laikui bėgant


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Išlaidų valdymo strategijos

Išlaidų kontrolė yra kritiškai svarbi gamybinėms AI agentėms. Štai pagrindinės strategijos:

| Strategy | Description |
|---|---|
| **Užklausų optimizavimas** | Laikykite sistemos nurodymus glaustus. Pašalinkite perteklinį kontekstą, kad sumažintumėte įvesties tokenų skaičių. |
| **Modelio pasirinkimas** | Naudokite mažesnius, pigesnius modelius (pvz. GPT-4o-mini) paprastoms užduotims, tokioms kaip klasifikacija ar išgavimas, o sudėtingesnį samprotavimą palikite didesniems modeliams. |
| **Talpyklavimas** | Talpykloje saugokite įrankių rezultatus ir dažnas užklausas, kad išvengtumėte pasikartojančių API iškvietimų. |
| **Tokenų biudžetai** | Nustatykite `max_tokens` ribas, kad išvengtumėte netikėtai ilgų atsakymų. |
| **Grupavimas** | Jei įmanoma, sugrupuokite kelias vartotojo užklausas į vieną API užklausą. |

Praktikoje sluoksniuota prieiga veikia gerai: nukreipkite paprastas užklausas į greitą, nebrangų modelį ir tik sudėtingesnes užklausas deleguokite galingesniam modeliui.


## Santrauka

Šioje pamokoje sužinojote, kaip:

1. **Pridėti matomumą** agentų sąveikoms naudojant laiko matavimą ir žurnalavimą, klojant pagrindą sekimui ir stebėsenai.
2. **Įvertinti agentų atsakymus** automatiškai naudojant vertinimo agentą, kuris įvertina išsamumą, tikslumą ir naudingumą.
3. **Valdyti išlaidas** per užklausų optimizavimą, modelio pasirinkimą, talpyklavimą ir žetonų biudžetus.

Šie gamybos modeliai padeda užtikrinti, kad jūsų dirbtinio intelekto agentai būtų patikimi, matuojami ir ekonomiškai efektyvūs dideliu mastu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba turėtų būti laikomas autoritetingu šaltiniu. Svarbios informacijos atveju rekomenduojamas profesionalus žmogaus vertimas. Mes neatsakome už jokius nesusipratimus ar neteisingus aiškinimus, kylančius dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
